In [1]:
!pip install pypdf faiss-cpu sentence-transformers langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("ft.pdf")
data = loader.load()
print(f"Loaded {len(data)} pages from the PDF")

Loaded 97 pages from the PDF


In [5]:
pip install langchain-text-splitters

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(data)
print(f"Created {len(chunks)} chunks")

Created 758 chunks


In [8]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceBgeEmbeddings(model_name = "all-miniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)
print("Vector db is ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-miniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector db is ready


In [9]:
query = "what is the accurate drichilet conditions"
docs = vector_db.similarity_search(query, k=3)
for i, doc in enumerate(docs):
  print(f"\n--- Chunk{i+1}---")
  print(doc.page_content[:400] + "...")


--- Chunk1---
and published in (2020) [3].
▶ Proper Definitions and Demonstration of Dirichlet Conditions...

--- Chunk2---
Dirichlet conditions (over entire period–at most a countably...

--- Chunk3---
pends on the type of used wavelet function, (ii) at low fre-...


In [10]:
!pip install -q U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.3 MB/s eta 0:00:00


In [11]:
import os
from langchain_openai import ChatOpenAI

# 1. Set your OpenRouter API Key
os.environ["OPENROUTER_API_KEY"] = "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"

# 2. Initialize the model
# The model ID for DeepSeek R1 on OpenRouter is usually 'deepseek/deepseek-r1'
# or 'deepseek/deepseek-r1:free' if you are using the free endpoint.
llm = ChatOpenAI(
    model="deepseek/deepseek-r1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/", # Optional
        "X-Title": "My Colab RAG App", # Optional
    }
)

In [12]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Explain the error in previous drichilet conditions.")])

# The actual answer
print("--- ANSWER ---")
print(response.content)

# To see the 'thinking' (if supported by the specific OpenRouter provider)
if "reasoning_content" in response.additional_kwargs:
    print("\n--- THINKING PROCESS ---")
    print(response.additional_kwargs["reasoning_content"])

--- ANSWER ---
The term "Dirichlet conditions" historically refers to two different contexts in mathematics, and "errors" in understanding them typically arise from **misinterpretations, oversimplifications, or misapplications**. Without knowing the exact "previous" conditions you're referring to, I’ll outline common areas where confusion or errors occur in each field:

---

### 1. **Dirichlet Conditions in Fourier Series**
These are **sufficient conditions** for a periodic function \( f(x) \) to be representable by a convergent Fourier series.

**Standard Dirichlet conditions:**
- \( f(x) \) is **absolutely integrable** over one period.
- \( f(x) \) has a **finite number of discontinuities** in one period.
- \( f(x) \) has a **finite number of extrema** in one period (i.e., it is of **bounded variation**).

**Common errors:**
- **Assuming necessity**: These conditions are **sufficient but not necessary**. There are functions that violate them yet still have convergent Fourier series.


In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [20]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

retriever = vector_db.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ✅ Prompt (THIS FIXES YOUR ERROR)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

{context}

Question: {question}
""")

# ✅ Fixed RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# run
result = rag_chain.invoke("What does the PDF say about Hilbert transform?")
print(result)

The PDF indicates that the Hilbert transform (HT) first appeared in David Hilbert's 1905 work. It also references a 1963 paper by J.L. Brown on a Hilbert transform product theorem published in the *Proceedings of the IEEE*. Additionally, the text associates the Hilbert transform with the Fourier and Phase transforms in a section authored by Pushpendra Singh, PhD (IIT Delhi), and cites F.W King's 2009 book *Hilbert Transforms* as a key source. The historical context and theoretical developments (e.g., the product theorem) are highlighted.
